In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import networkx as nx
import json
from pathlib import Path
from collections import Counter
from ipysigma import Sigma

import sys
import re
sys.path.append('../')
from source import Align
from textwrap import wrap

## EDA

In [ ]:
BlastPairWiseAlignmentPivoted = pd.read_csv("../data/processed/BlastPairWiseAlignmentPivoted.cov80.maxseq1.csv", sep="\t")
mask = np.triu(np.ones_like(BlastPairWiseAlignmentPivoted.set_index("Unnamed: 0"), dtype=bool)) 
sns.heatmap(
    BlastPairWiseAlignmentPivoted.set_index("Unnamed: 0"),
    cmap="Blues",
    annot=True,
    fmt=".0f",
    linewidth=.5,
    mask=mask,
)
plt.savefig(f"../figures/DatabasesHeatmap.svg", 
            format="svg", 
            dpi=180, 
            bbox_inches='tight')

In [ ]:
BlastPairWiseAlignmentPivoted

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 3))
ax1.set_xlabel("Maximum Identity Threshold (%)")
ax1.set_ylabel('Database Size')

ClustersCdHit = pd.read_csv(
    "../data/processed/cdhitclusters.defaultsettings.csv",
    sep = "\t",
)
ax1.plot(
    ClustersCdHit.set_index("Unnamed: 0"),
    marker = "o",
    linestyle='--',
    linewidth=0.8
)
ax1.grid(True, linestyle='--', alpha=0.5)
plt.savefig(f"../figures/ClusterSizes.svg", 
            format="svg", 
            dpi=180, 
            bbox_inches='tight')

In [ ]:
ClustersCdHit

In [ ]:
CARDLOG = pd.read_csv("../data/processed/Card-Changelog.tsv",sep = "\t")
CARDLOG_CHAGNES = CARDLOG[["Changes","Date"]].groupby("Date").sum().reset_index()
len(CARDLOG)

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 3))
color_blue = '#1f77b4'
ax1.set_xlabel('Date (Year)')
ax1.set_ylabel('Total of Proteins', color=color_blue)

line1 = ax1.plot(CARDLOG['Date'], CARDLOG['Total of Proteins'], color=color_blue, marker='o',linestyle='--', linewidth=1, label='Total Of Proteins')
ax1.tick_params(axis='y', labelcolor=color_blue)
ax1.grid(True, linestyle='--', alpha=0.5)
ax1.set_ylim(0, 7000)

ax2 = ax1.twinx()
color_orange = '#ff7f0e'
ax2.set_ylabel('Changes', color=color_orange)
line2 = ax2.plot(CARDLOG_CHAGNES['Date'], CARDLOG_CHAGNES['Changes'], color=color_orange, marker='s', linestyle='--', linewidth=1, label='Changes')
ax2.tick_params(axis='y', labelcolor=color_orange)

ax1.tick_params(axis='x', rotation=90)

plt.savefig(f"../figures/CARD-Curve.svg", 
            format="svg", 
            dpi=180, 
            bbox_inches='tight')

In [ ]:
PairWiseAlignment = pd.read_csv(
    "../data/filtered/AllDatabases.Paiwise.cov80.maxseq1.tsv", 
    sep = "\t",
    skipinitialspace=True, 
    header=None,
    names = "qseqid sseqid stitle pident length qlen slen qstart qend sstart send evalue bitscore ppos full_qseq full_sseq".split(" ")
)
PairWiseAlignment["qcov"] = np.round((PairWiseAlignment["qend"] - PairWiseAlignment["qstart"] + 1) / (PairWiseAlignment["qlen"]) * 100, 2)
PairWiseAlignment["scov"] = np.round((PairWiseAlignment["send"] - PairWiseAlignment["sstart"] + 1) / (PairWiseAlignment["slen"]) * 100, 2)
PairWiseAlignment["qseqtag"] = PairWiseAlignment["qseqid"].str.split("|").str[1]
PairWiseAlignment["sseqtag"] = PairWiseAlignment["sseqid"].str.split("|").str[1]
PairWiseAlignment["qseqid"] = PairWiseAlignment["qseqid"].str.split("|").str[0]
PairWiseAlignment["sseqid"] = PairWiseAlignment["sseqid"].str.split("|").str[0]

In [ ]:
PairWiseAlignment

In [ ]:
sns.scatterplot(
    data=PairWiseAlignment.loc[PairWiseAlignment["bitscore"] > 70], 
    x="qcov", 
    y="pident",
    size = "bitscore",
    )

In [ ]:
sns.scatterplot(
    data=PairWiseAlignment.loc[PairWiseAlignment["bitscore"] > 80], 
    x="pident", 
    y="ppos",
    size = "bitscore",
    )

In [ ]:
fig, ax1 = plt.subplots(figsize=(5, 5))
ax1.set_xlabel("Protein Identity")
ax1.hist(PairWiseAlignment.pident)
ax1.grid(True, linestyle='--', alpha=0.5)
plt.savefig(f"../figures/DiamondIdentityDistribution-PairwiseAlign.svg", 
            format="svg", 
            dpi=180, 
            bbox_inches='tight')

In [ ]:
with open("../data/processed/MetaDict.cov80.maxseq1.json", "r") as json_file  :
    MetaDict = json.load(json_file)

## Networks

In [ ]:
SequenceSimilarityGraph = nx.from_pandas_edgelist(
    PairWiseAlignment.loc[PairWiseAlignment["pident"] > 95], 
    source="qseqid", 
    target="sseqid", 
    edge_attr=["pident", "evalue","bitscore", "ppos"]
)
nx.set_node_attributes(SequenceSimilarityGraph, MetaDict)
ConnectedComponents = list(nx.connected_components(SequenceSimilarityGraph))
len(ConnectedComponents)

In [ ]:
ProblematicComponents = []
for component in nx.connected_components(SequenceSimilarityGraph):
   DrugClasses = {SequenceSimilarityGraph.nodes[node].get("Drug Class") for node in component}
   if len(DrugClasses) > 1:
      ProblematicComponents.extend(component)
ProblematicGraph = SequenceSimilarityGraph.subgraph(ProblematicComponents).copy()

In [ ]:
ClassCount = []
for component in nx.connected_components(ProblematicGraph):
    DrugClassesLen = {ProblematicGraph.nodes[node].get("Drug Class") for node in component}
    ClassCount.append(DrugClassesLen)

SetCount = Counter(frozenset(sorted(s)) for s in ClassCount)

SetCountList = []
for Set,Total in SetCount.items():
    label = "{"+", ".join(sorted(list(Set)))+"}"
    SetCountList.append({"Set":label, "Total":Total})
SetCount = pd.DataFrame(SetCountList)
SetCount.sort_values(by = "Total", ascending = False)
SetCount.to_csv("../tables/ProblematicClassesCount.tsv", sep = "\t", index = False)

plt.figure(figsize=(2, 4))
sns.barplot(
    data=SetCount.sort_values(
        by = "Total", ascending = False).head(15), 
        x='Total', 
        y='Set', 
        color ='lightgrey')

plt.xlabel('Count')
plt.ylabel('Sets')
plt.tight_layout() 
plt.savefig("../figures/Top30ProblematicSetsCount.svg", 
            format="svg", 
            dpi=180, 
            bbox_inches='tight')


FinalClassCount = Counter([classe for sets in ClassCount for classe in sets])
pd.Series(FinalClassCount).sort_values(ascending=False).head(30).to_frame().plot(kind = "barh", figsize=(1,7), color ='lightgrey')
plt.xlabel('Count')
plt.ylabel('Classes')
plt.tight_layout() 
plt.savefig("../figures/Top30ProblematicClassesCount.svg", 
            format="svg", 
            dpi=180, 
            bbox_inches='tight')



In [ ]:
for TargetClass in pd.Series(dict(FinalClassCount)).sort_values(ascending=False).head(30).index:
    # if pd.Series(dict(FinalClassCount)).sort_values(ascending=False).head(30).index != "GenomeProtein":
        SetCount.loc[SetCount["Set"].str.contains(TargetClass)].head(10).set_index("Set").sort_values("Total",ascending = False).plot(kind = "bar", figsize=(4,1), color ='lightgrey')
        plt.ylabel('Count')
        plt.xlabel('Sets')
        plt.title(TargetClass.capitalize())
        plt.savefig(f"../figures/InterestingProblematicSets.{TargetClass.replace('/','-')}.svg", 
                    format="svg", 
                    dpi=180, 
                    bbox_inches='tight')

In [ ]:
color_map = {
    'multidrug':      "#FD6F63",     
    'beta-lactam':    "#89BDE7",  
    'bacitracin':     "#2E8D19",    
    'tetracycline':   "#CD73E8",
    'glycopeptide':   "#F3BC71",  
    'macrolide':      "#D4D451",     
    'aminoglycoside': "#ECAD2F", 
    'quinolone':      "#14E974",
    'drug_and_biocide_resistance': "#EA17B9",
    "drug_and_biocide_and_metal_resistance": "#EA17B9",
    'fosfomycin':     "#6063F4",
    'fluoroquinolone':"#60D4F4",
    'elfamycin':      "#7B079E",
    'aminocoumarin':  "#EE00F6",
    'rifampin':       "#0BF8E8",
    'phosphonic acid':"#262897",
    'fosmidomycin':   "#7B5404",
    'colistin' :      "#7FBE03",
    'polymyxin':      "#97B262",
    'GenomeProtein':"#534F51",
    }

In [30]:
outdir = Path("../data/processed/selected_components")
outdir.mkdir(parents=True, exist_ok=True)

for TargetClass in color_map.keys():
    if TargetClass != "GenomeProtein":
        
        FilteredNodes = [nx.node_connected_component(ProblematicGraph, Node) for Node, Data in ProblematicGraph.nodes(data=True) if Data.get("Drug Class") == TargetClass]
        FilteredNodes = [nodes for component in FilteredNodes for nodes in component]


        SelectedSubGraph = ProblematicGraph.subgraph(FilteredNodes)
        print(TargetClass.replace(' ','_'))
        nx.write_graphml(SelectedSubGraph, f"../data/processed/ProblematicComponentsByClass.{TargetClass.replace(' ','_')}.graphml")
        with open(f"../data/processed/selected_components/ProblematicComponentsByClass.{TargetClass.replace(' ','_')}.fasta", "w+") as fastafile:
            for Node, Data in SelectedSubGraph.nodes(data=True):
                fastafile.write(f">{Node}|{Data.get('Drug Class')}|{Data.get('Name')}\n{Data.get('Sequence')}\n")
            


multidrug
beta-lactam
bacitracin
tetracycline
glycopeptide
macrolide
aminoglycoside
quinolone
drug_and_biocide_resistance
drug_and_biocide_and_metal_resistance
fosfomycin
fluoroquinolone
elfamycin
aminocoumarin
rifampin
phosphonic_acid
fosmidomycin
colistin
polymyxin


In [ ]:
nx.node_connected_component(ProblematicGraph, 'CARD_2')

In [ ]:
FilteredNodes

In [ ]:
meu_layout = {
    "scalingRatio": 50.0,           # Aumente para afastar os grupos
    "gravity": 0.2,                 # Reduza para não amontoar no centro
    "repulsion": 2,               # Aumente para afastar os nós
    # "outboundAttractionDistribution": True, # Empurra hubs para fora
    # "barnesHutOptimize": True,      # Essencial para seus 70k nós
    # "linLogMode": True              # Melhora a definição de clusters biológicos 
}

Sigma(
    ProblematicGraph, 
    node_size  = ProblematicGraph.degree(), 
    node_color =  [ProblematicGraph.nodes[n].get("Drug Class") for n in ProblematicComponentsGraph.nodes()],
    raw_node_color=True,
    node_color_palette = color_map,
    # node_metrics={"community": "louvain"},

    default_edge_type = "curve",
    layout_settings=meu_layout,
    start_layout=30,
    )

In [ ]:
TargetClass = color_map.keys()
TargetClassSet = set()
for key, value in ComponentIndex.items():
    if TargetClass in value["classes"]:
        TargetClassSet = TargetClassSet.union(value["component"])
ProblematicComponentsByClass = SequenceSimilarityGraph.subgraph(TargetClassSet)
# nx.write_graphml(ProblematicComponentsByClass, f"../data/processed/ProblematicComponentsByClass.{TargetClass}.graphml")


meu_layout = {
    "scalingRatio": 50.0,           # Aumente para afastar os grupos
    "gravity": 0.2,                 # Reduza para não amontoar no centro
    "repulsion": 2,               # Aumente para afastar os nós
    # "outboundAttractionDistribution": True, # Empurra hubs para fora
    # "barnesHutOptimize": True,      # Essencial para seus 70k nós
    # "linLogMode": True              # Melhora a definição de clusters biológicos 
}
Sigma(
    ProblematicComponentsByClass, 
    node_size  = ProblematicComponentsByClass.degree(), 
    node_color =  [ProblematicComponentsByClass.nodes[n].get("Drug Class") for n in ProblematicComponentsByClass.nodes()],
    raw_node_color=True,
    node_color_palette = color_map,
    # node_metrics={"community": "louvain"},

    default_edge_type = "curve",
    layout_settings=meu_layout,
    start_layout=30,
    )

In [ ]:
outdir = Path("../data/processed/selected_components")
outdir.mkdir(parents=True, exist_ok=True)

TargetClass = color_map.keys()
TargetClassSet = set()
for TargetClass in color_map.keys():
    for key, value in ComponentIndex.items():
        if TargetClass in value["classes"] and TargetClass != "GenomeProtein":
            TargetClassSet = TargetClassSet.union(value["component"])
            ProblematicComponentsByClass = SequenceSimilarityGraph.subgraph(TargetClassSet)
            print(TargetClass)
            print(ProblematicComponentsByClass.nodes)


In [ ]:
list(nx.connected_components(ProblematicComponents))

In [ ]:
outdir = Path("../data/processed/selected_components")
outdir.mkdir(parents=True, exist_ok=True)

for TargetClass in color_map.keys():
    if TargetClass != "GenomeProtein":
        print(TargetClass.replace(' ','_'))
        nx.write_graphml(ProblematicComponentsByClass, f"../data/processed/ProblematicComponentsByClass.{TargetClass.replace(' ','_')}.graphml")
        with open(f"{outdir}/ProblematicComponents.TargetClass.{TargetClass.replace(' ','_')}.fasta", "w+") as ProblematiComponentsFastaFile:
            for ProteinNode in ProblematicComponentsByClass.nodes:
                try:
                    NodeClass = ProblematicComponentsByClass.nodes[ProteinNode]["Drug Class"]
                    NodeSequence = ProblematicComponentsByClass.nodes[ProteinNode]["Sequence"]
                except:
                    pass
                ProblematiComponentsFastaFile.write(f">{ProteinNode}|{NodeClass}\n{NodeSequence}\n")

In [ ]:
outdir = Path("../data/processed/selected_components")
outdir.mkdir(parents=True, exist_ok=True)


with open(f"{outdir}/ProblematicComponents.TargetClass.{TargetClass}.fasta", "w+") as ProblematiComponentsFastaFile:
    for ProteinNode in ProblematicComponentsByClass.nodes:
        try:
            NodeClass = ProblematicComponentsByClass.nodes[ProteinNode]["Drug Class"]
            NodeSequence = ProblematicComponentsByClass.nodes[ProteinNode]["Sequence"]
        except:
            pass
        ProblematiComponentsFastaFile.write(f">{ProteinNode}|{NodeClass}\n{NodeSequence}\n")
        # print(f">{ProteinNode}|{NodeClass}\n{NodeSequence}\n")

In [ ]:
ProblematicCompoLenDist = [value["len"] for key, value in ComponentIndex.items() if value["status"] == "problematic"]

In [ ]:
plt.figure(figsize=(1,4))
sns.violinplot(ProblematicCompoLenDist)
plt.ylabel("Problematic Connected Components Sizes")
plt.savefig(f"../figures/DisProblematicComponentSizes.svg", 
            format="svg", 
            dpi=180, 
            bbox_inches='tight')

In [ ]:
ProblematicCompoClassDist = [len(value["classes"]) for key, value in ComponentIndex.items() if value["status"] == "problematic"]

In [ ]:
plt.figure(figsize=(1,4))
sns.violinplot(ProblematicCompoClassDist)
plt.ylabel("Number of classes")
plt.savefig(f"../figures/DisClassNumber.svg", 
            format="svg", 
            dpi=180, 
            bbox_inches='tight')

In [ ]:
def ConcateComponentsBy(CompDict, OriginalGraph, Minsize = None, MinClassNumber = None):
    if Minsize is None and MinClassNumber is None:
        raise ValueError("At least one of Minsize or MinClassNumber must be provided. Both cannot be None.")
    elif Minsize is not None and MinClassNumber is not None:
        raise ValueError("Only one of Minsize or MinClassNumber can be provided. Both cannot be set at the same time.")
    
    ConcateComponents = set()
    for key, value in CompDict.items():
        if Minsize is not None and value["len"] >= Minsize and value["status"] == "problematic":
            ConcateComponents = ConcateComponents.union(value["component"])

        elif MinClassNumber is not None and len(value["classes"]) >= MinClassNumber and value["status"] == "problematic":
            ConcateComponents = ConcateComponents.union(value["component"])            
    return OriginalGraph.subgraph(ConcateComponents),ConcateComponents

In [ ]:
ProblematicComponenetBy,_ = ConcateComponentsBy(ComponentIndex,SequenceSimilarityGraph,Minsize=20)
[len(comp) for comp in list(nx.connected_components(ProblematicComponenetBy))]

In [ ]:
meu_layout = {
    "scalingRatio": 50.0,           # Aumente para afastar os grupos
    "gravity": 0.3,                 # Reduza para não amontoar no centro
    "repulsion": 3,               # Aumente para afastar os nós
    # "outboundAttractionDistribution": True, # Empurra hubs para fora
    # "barnesHutOptimize": True,      # Essencial para seus 70k nós
    # "linLogMode": True              # Melhora a definição de clusters biológicos 
}
Sigma(
    ProblematicComponenetBy, 
    node_size  = ProblematicComponenetBy.degree(), 
    node_color =  [ProblematicComponenetBy.nodes[n].get("Drug Class") for n in ProblematicComponenetBy.nodes()],
    raw_node_color=True,
    node_color_palette = color_map,
    # node_metrics={"community": "louvain"},

    default_edge_type = "curve",
    layout_settings=meu_layout,
    start_layout=30,
    )

In [ ]:
import matplotlib.patches as mpatches
legend_elements = [
    mpatches.Patch(color=color, label=label) 
    for label, color in color_map.items()
]

# Configurando uma figura vazia apenas para conter a legenda
fig, ax = plt.subplots(figsize=(4, 6))
ax.axis('off') # Esconde os eixos do gráfico

# Desenhando a legenda centralizada
ax.legend(
    handles=legend_elements, 
    loc='center', 
    frameon=True, 
    facecolor='white', 
    edgecolor='none',
    fontsize=16,
    title="Drug Classes",
    title_fontsize=13
)

# Salvando a legenda de forma que as bordas extras sejam cortadas
plt.savefig("../figures/Network_Legend.svg", format="svg", bbox_inches='tight', dpi=180)
plt.show()